# Suite TS5PRS — Parsing TypeScript 5 into the store

`TypeScript5.Parse.parse(text, path)` reads TypeScript 5 source with tree-sitter-typescript, with its `tsx` grammar where
the path ends `.tsx`, and gives the callback of the transaction that writes it: one upsert, binding the workspace and the
program, that files the program in the workspace by its path and writes the program's `body` as one value. The whole
tree is values: each statement is a value object, a branch of `Programs.TypeScript5.Statement` tagged by its kind in
lower case, with the line it begins on (`line`), holding the statements of its blocks, its clauses and its class
members in its properties, in order. Expressions are mbse-expressions' `TypeScript5` terms, types `TypeScript Types`
terms and binding patterns `TypeScript Patterns` terms, each keeping the concrete syntax the dialect holds; where a term
holds statements, its `block` or `classBody` holds them. Each case parses constructs into one store, reads the
program's tree back as plain data within a transaction (a term as `ToText` writes it, or as its kind, attributes and
subterms, its block or class body as plain data among its attributes), and checks it; then invalid syntax, which is
kept, and the one error, text that is no text, with its line and column. Its cases port 0.3's TSPRS.

In [ ]:
import itertools
import json
from pathlib import Path

from mbse.Expressions.Framework import Objects, Terms
from mbse.Expressions.TypeScript.TypeScript5 import Expressions as T
from mbse.Expressions.TypeScript.TypeScript5.Text import ToText
from mbse.Programs import TypeScript5 as Programs
from mbse.Programs.Framework import Model
from mbse.Programs.Framework.Errors import ParseError
from mbse.Programs.TypeScript5 import Parse
from mbse.Programs.TypeScript5.Statements import KINDS, PARTS
from mbse.Schemas.Framework import Plain, References, Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Programs.store()
PATHS = (f"case{n}" for n in itertools.count())
HOLDERS = ("block", "classBody")  # a term's properties that hold this grammar's statements


def shape(term):
    """A term as plain data: its kind, its attributes (a helper as its kind and attributes, its block or class body as
    plain data), and its subterms."""
    def part(value):
        if isinstance(value, Terms.Part):
            return (value.kind, {k: part(v) for k, v in value.attributes.items()})
        return [part(v) for v in value] if isinstance(value, tuple) else value
    tag, record = Terms.kind_of(term)
    attributes = {k: part(v) for k, v in Terms.attributes_of(record).items()}
    held = Objects.properties(record)
    attributes.update({name: plain(held[name], True) for name in HOLDERS if name in held})
    return (tag, attributes, [None if a is None else shape(a) for a in Terms.arguments_of(record)])


def plain(value, shapes=False):
    """What a property holds, as plain data: a term as its text (or its `shape`), a container as a list, a union's value
    as its branch's, a value object as a dict of its kind and its properties."""
    if isinstance(value, Objects.Listed):
        return [plain(v, shapes) for v in value.values]
    if Terms.is_term(value):
        return shape(value) if shapes else ToText(value)
    if Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        return plain(Terms.kind_of(value)[1], shapes)
    if Objects.is_object(value):
        return {"kind": value.as_schema().name.rsplit(".", 1)[-1]} | {
            k: plain(v, shapes) for k, v in Objects.properties(value).items()}
    return value


def program(store, path):
    """The program at `path`, read within the transaction open on `store`."""
    return store.singleton(Model.WORKSPACE).files({"path": path})


def parse(text, shapes=False, suffix=".ts"):
    """The statements of the program `text` parses to, written in the store at a path of its own, read back."""
    path = next(PATHS) + suffix
    STORE.transaction(Parse.parse(text, path))
    with STORE.transaction():
        return plain(Objects.properties(program(STORE, path))["body"], shapes)


def first(text, shapes=False, suffix=".ts"):
    return parse(text, shapes, suffix)[0]


def value(text, shapes=False, suffix=".ts"):
    """The expression of the expression statement `text`."""
    return first(text, shapes, suffix)["expression"]


def tsx(text, shapes=False):
    """The expression of the expression statement `text`, in a `.tsx` file."""
    return value(text, shapes, ".tsx")


def kinds(nodes):
    return [n["kind"] for n in nodes]


def assigned(store, frame):
    """What `frame`, given back by a transaction on `store`, assigns, as `References.OfFrame` writes it as plain data."""
    written = Plain.ToPlain(store).Value(store.registered("References.OfFrame"), References.of(store, frame))
    return written["assigns"]


def fails(text, message, line, column):
    with raises(ParseError, match=message):
        Parse.parse(text, "failing.ts")
    try:
        Parse.parse(text, "failing.ts")
    except ParseError as error:
        assert (error.line, error.column) == (line, column), (error.line, error.column)

## TS5PRS-01 · The conformance sources are one transaction each: every statement, clause and member kind and every part, terms their dialects validate, and JSON that reads back and writes the same bytes

In [ ]:
def walk(value, outermost=True):
    """The value objects and the outermost terms within `value`, in source order: a term's subterms and helpers are its
    own, but the statements its block or class body holds are walked as any."""
    if isinstance(value, Objects.Listed):
        for v in value.values:
            yield from walk(v, outermost)
    elif Terms.is_term(value):
        if outermost:
            yield value
        for name, held in Objects.properties(Terms.kind_of(value)[1]).items():
            yield from walk(held, name in HOLDERS)
    elif Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        yield from walk(Terms.kind_of(value)[1], outermost)
    elif Objects.is_object(value):
        if outermost:
            yield value
        for name, held in Objects.properties(value).items():
            yield from walk(held, outermost or name in HOLDERS)


seen, problems, counts = set(), [], {}
for name, statements in (("typescript59.ts", 111), ("typescript59-jsx.tsx", 6)):
    text = (SOURCES / name).read_text(encoding="utf-8")
    (frame,) = STORE.transaction(Parse.parse(text, name))
    assert sorted(frame) == ["program", "ws"]  # the workspace and the program; every statement is a value within it
    (assign,) = assigned(STORE, frame)  # the program, given its whole tree as one nested value
    assert len(assign["properties"]["body"]["Programs.TypeScript5.Body"]) == statements, name
    with STORE.transaction():
        held = Objects.properties(program(STORE, name))
        assert held.get("hashbang") == ("/usr/bin/env node" if name.endswith(".ts") else None)
        for node in walk(held["body"]):
            if Terms.is_term(node):  # names are bound by the program, not the term: only other problems count
                problems.extend(p for p in T.DIALECT.validate(node) if "is not bound" not in p)
            else:
                seen.add(node.as_schema().name.rsplit(".", 1)[-1])
                counts[name] = counts.get(name, 0) + 1
    # Written as plain JSON, read into a fresh store holding the installs, executed, and written again: the same bytes
    written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
        Parse.parse(text, name, STORE))))
    fresh = Programs.store()
    (again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
    assert sorted(again) == sorted(frame) and assigned(fresh, again) == assigned(STORE, frame)
    assert json.dumps(Plain.ToPlain(fresh).Transaction(fresh.Mutations.OfTransaction(
        Parse.parse(text, name, fresh)))) == written
assert set(KINDS) <= seen, set(KINDS) - seen  # every statement, clause and member kind
assert set(PARTS) <= seen, set(PARTS) - seen  # and every part that holds no statement
assert problems == [], problems[:5]
print(len(seen), "kinds,", counts)

## TS5PRS-02 · The transaction: one upsert by the file's path, binding the workspace and the program, the program filed by its path, its tree one nested value, each statement in its parent's property, in order, with its line, a term's statements in its `block`

In [ ]:
source = "if (a) {\n} else {\n    b = 1;\n}\nlet f = () => {\n    return 1;\n};\n"
(frame,) = STORE.transaction(Parse.parse(source, "shape.ts"))
assert sorted(frame) == ["program", "ws"]
assert set(frame.assigns) == {"program"} and set(frame.assigns["program"]) == {"body"}
(assign,) = assigned(STORE, frame)  # the program's tree, as one value, its statements nested in source order
one = {"literal": {"value": {"float": 1.0}, "text": "1", "base": 10}}
assert assign["properties"] == {"body": {"Programs.TypeScript5.Body": [
    {"ifstatement": {"line": 1, "test": {"identifier": {"name": "a"}}, "consequent": {"blockstatement": {"line": 1}},
                     "alternate": {"blockstatement": {"line": 2, "body": [{"expressionstatement": {
                         "line": 3, "expression": {"assignmentexpression": {
                             "left": {"identifier": {"name": "b"}}, "operator": "=", "right": one}}}}]}}}},
    {"variabledeclaration": {"line": 5, "declarationKind": "let", "declarations": [{
        "id": {"identifier": {"name": "f"}}, "init": {"arrowfunctionexpression": {"block": {"blockstatement": {
            "line": 5, "body": [{"returnstatement": {"line": 6, "argument": one}}]}}}}}]}}]}}
with STORE.transaction():
    file = program(STORE, "shape.ts")
    assert file.as_schema().name == "Programs.TypeScript5.Program"
    top, declaration = Objects.properties(file)["body"].values  # the program's body, its statements
    tag, node = Terms.kind_of(top)
    assert (tag, node.as_schema().name) == ("ifstatement", "Programs.TypeScript5.IfStatement")
    held = Objects.properties(node)
    assert [Terms.kind_of(held[name])[0] for name in ("consequent", "alternate")] == ["blockstatement"] * 2
    (statement,) = Objects.properties(Terms.kind_of(held["alternate"])[1])["body"].values
    assert Objects.properties(Terms.kind_of(statement)[1])["line"] == 3
    # an arrow, a term, holds its statements in its `block`: this grammar's `BlockStatement`, a branch of the
    # dialect's open union `Expressions.TypeScript.TypeScript5.Block`
    (declarator,) = Objects.properties(Terms.kind_of(declaration)[1])["declarations"].values
    arrow = Objects.properties(declarator)["init"]
    assert Terms.kind_of(arrow)[0] == "arrowfunctionexpression"
    block = Objects.properties(Terms.kind_of(arrow)[1])["block"]
    assert block.as_schema().name == "Expressions.TypeScript.TypeScript5.Block"
    tag, node = Terms.kind_of(block)
    assert (tag, node.as_schema().name) == ("blockstatement", "Programs.TypeScript5.BlockStatement")
    (returned,) = Objects.properties(node)["body"].values
    assert Terms.kind_of(returned)[0] == "returnstatement"
# Run again, the upsert finds what it wrote
STORE.transaction(Parse.parse(source, "shape.ts"))
with STORE.transaction():
    assert len(Objects.properties(program(STORE, "shape.ts"))["body"].values) == 2
# The callback names its schemas through what store.transaction gives it; a bare builder needs the store
with raises(TypeError):
    STORE.Mutations.OfTransaction(Parse.parse(";", "bare.ts"))
built = Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse(";", "bare.ts", STORE)))
assert "bare.ts" in json.dumps(built)
assert parse("") == [] and kinds(parse("x\n\n\ny\n")) == ["ExpressionStatement", "ExpressionStatement"]
assert [s["line"] for s in parse("x = 1;\n\n@d\nclass C {}\n")] == [1, 3]  # a class's line, its decorator's
STORE.transaction(Parse.parse("#!/usr/bin/env node\nx;\n", "hashbang.ts"))
with STORE.transaction():
    assert Objects.properties(program(STORE, "hashbang.ts"))["hashbang"] == "/usr/bin/env node"
    assert "hashbang" not in Objects.properties(program(STORE, "shape.ts"))
print("ok")

## TS5PRS-03 · Literals keep their spelling: numbers, bigints, strings and their escapes, booleans, `null` and regular expressions, and templates' raw text

In [ ]:
def literal(text):
    tag, attributes, _ = value(text, shapes=True)
    return tag, attributes


assert literal("0xF_F") == ("literal", {"value": 255.0, "text": "0xF_F", "base": 16, "separators": True})
assert [literal(t)[1]["base"] for t in ("0o17", "0b1010", "0XF", "10", "1_000")] == [8, 2, 16, 10, 10]
assert literal("1.5e-3") == ("literal", {"value": 0.0015, "text": "1.5e-3"})  # spelled as no integer: no base
assert literal(".5")[1]["value"] == 0.5 and literal("5.")[1]["value"] == 5.0
assert literal("10n") == ("literal", {"value": 10, "text": "10n", "base": 10})  # a bigint: an int
assert literal("0xFF_FFn") == ("literal", {"value": 65535, "text": "0xFF_FFn", "base": 16, "separators": True})
assert literal("1" + "0" * 400)[1]["value"] == float("inf")  # beyond the largest number: `Infinity`, as JavaScript
assert literal("1" + "0" * 400 + "n")[1]["value"] == 10 ** 400  # a bigint, unbounded
assert literal("true") == ("literal", {"value": True, "text": "true"})
assert literal("false") == ("literal", {"value": False, "text": "false"})
assert literal("null") == ("constant", {"singleton": "null", "text": "null"})
assert literal("/a+/gi") == ("constant", {"pattern": "a+", "flags": "gi", "text": "/a+/gi"})
assert literal("/[/]/") == ("constant", {"pattern": "[/]", "text": "/[/]/"})
assert value("undefined", shapes=True) == ("identifier", {"name": "undefined"}, [])
assert literal("'a'") == ("literal", {"value": "a", "text": "'a'", "quote": "'"})
assert literal('"a"')[1]["quote"] == '"'
escaped = r"'\n\t\r\b\f\v\0\x41\u00e9\u{1F600}\uD83D\uDE00\q\'\\'"
assert literal(escaped)[1]["value"] == "\n\t\r\b\f\v\0A\u00e9\U0001F600\U0001F600q'\\"  # a pair joined
assert literal("'\\uD83D'")[1]["value"] == "\ud83d"  # a lone surrogate, as written
assert literal("'a\\\nb'")[1]["value"] == "ab"  # a line continued
for text in ("0", "1_000", "0xFF", "0o17", "0b1010", "1.5e-3", ".5", "5.", "10n", "0xFFn", "'a'", '"a"', escaped,
             "true", "false", "null", "/a+/gi", "/[/]/v", "undefined"):
    assert value(text) == text, (text, value(text))
template = value("`a${b}c${`d${e}`}`", shapes=True)
assert template[0] == "templateliteral" and template[1]["quasis"] == [
    ("templateelement", {"raw": "a"}), ("templateelement", {"raw": "c"}), ("templateelement", {"raw": "", "tail": True})]
assert [t[0] for t in template[2]] == ["identifier", "templateliteral"]
assert [q[1]["raw"] for q in value("`\\n${x}\\``", shapes=True)[1]["quasis"]] == ["\\n", "\\`"]
tagged = value("String.raw`x${y}`", shapes=True)
assert tagged[0] == "taggedtemplateexpression" and tagged[1]["quasi"][0] == "templateliteral"
for text in ("`a${b}c`", "String.raw`x${y}`", "`\\n${x}\\``"):
    assert value(text) == text, (text, value(text))
print("ok")

## TS5PRS-04 · Expressions, with TypeScript's grouping where tree-sitter-typescript's differs: `as`, `satisfies` and `!` regrouped, `x as A.B.C`'s names the type's

In [ ]:
def tree(text):
    return value(text, shapes=True)


grouped = tree("a ?? b as T")  # `as` binds tighter than `??`
assert grouped[0] == "logicalexpression" and grouped[2][1][0] == "tsasexpression"
grouped = tree("a || b && c as number < d")
assert (grouped[1]["operator"], grouped[2][1][1]["operator"], grouped[2][1][2][1][1]["operator"]) == ("||", "&&", "<")
assert grouped[2][1][2][1][2][0][0] == "tsasexpression"
assert tree("a || b as T && c")[2][1][1]["operator"] == "&&"  # `&&` moves onto `as`'s operand too
assert tree("a ? b : c as T")[2][2][0] == "tsasexpression"
assert tree("x = a satisfies T")[2][1][0] == "tssatisfiesexpression"
assert tree("f = () => a as T")[2][1][2][1][0] == "tsasexpression"  # the arrow's body
assert tree("a == b as T == c")[2][0][1]["operator"] == "=="
assert value("x as A.B.C") == "x as A.B.C" and tree("x as A.B.C")[0] == "tsasexpression"  # the names are the type's
assert value("x as A | B.C.D") == "x as A | B.C.D" and value("x satisfies A & B.C.D") == "x satisfies A & B.C.D"
assert tree("x as A | B.C.D")[2][1][0] == "tsuniontype"
assert tree("(x as T).y")[0] == "memberexpression" and tree("x as T[]")[0] == "tsasexpression"
assert tree("x as T[].y")[0] == "memberexpression"  # only names continue a type
assert tree("[1] as const")[2][1] == ("tstypereference", {}, [("identifier", {"name": "const"}, [])])
assert tree("a ?? b as const")[2][1][0] == "tsasexpression" and tree("a + b as T")[0] == "tsasexpression"
asserted = tree("a ?? b!")
assert asserted[0] == "logicalexpression" and asserted[2][1][0] == "tsnonnullexpression"
assert tree("a ? b : c!")[2][2][0] == "tsnonnullexpression" and tree("x = y!")[2][1][0] == "tsnonnullexpression"
chain = tree("a?.b.c!.d?.[e]?.(f)")
assert chain[0] == "chainexpression" and chain[2][0][0] == "callexpression" and chain[2][0][1]["optional"]
assert tree("a!.b")[0] == "memberexpression" and tree("a.b")[0] == "memberexpression"
assert tree("(a?.b).c")[2][0][0] == "parenthesizedexpression"
assert [t[0] for t in tree("a, b, c")[2]] == ["identifier"] * 3
assert tree("x++")[1] == {"operator": "++"} and tree("--x")[1] == {"operator": "--", "prefix": True}
assert tree("f<T>(a)")[1]["typeArguments"][0] == "tstypeparameterinstantiation"
assert tree("f<T>")[0] == "tsinstantiationexpression"
assert tree("new A<T>")[0] == "newexpression" and tree("new A")[2] == [("identifier", {"name": "A"}, [])]
imported = tree("import('m', { with: {} })")
assert imported[0] == "importexpression" and [t[0] for t in imported[2]] == ["literal", "objectexpression"]
assert tree("import('m')")[2][1] is None and tree("<T>x")[0] == "tstypeassertion"
assert tree("#x in o")[2][0] == ("privateidentifier", {"name": "x"}, [])
assert tree("a **= b")[1]["operator"] == "**=" and tree("a >>>= b")[1]["operator"] == ">>>="
assert tree("yield* g")[1] == {"delegate": True} and tree("yield")[2] == [None]
arrow = tree("async (a) => { await a; }")
assert arrow[1]["isAsync"] and arrow[1]["block"]["body"][0]["expression"][0] == "awaitexpression"
assert tree("a => a")[0] == "arrowfunctionexpression" and tree("a => a")[2][1:] == [("identifier", {"name": "a"}, [])] * 2
assert [t[0] for t in tree("[a, , ...b]")[2]] == ["identifier", "elision", "spreadelement"]
literal = tree("({ a, b: 1, [c]: 2, 'd': 3, 4: 5, m() {}, get g() { return 1; }, set s(v) {}, async *k() {}, ...r })")
properties = literal[2][0][2]
assert [p[1].get("propertyKind") for p in properties] == ["init"] * 6 + ["get", "set", "init", None]
assert properties[0][1]["shorthand"] and properties[2][1]["computed"] and properties[5][1]["method"]
assert properties[6][2][1][1]["block"]["body"][0]["kind"] == "ReturnStatement"  # a method's statements, its block
assert properties[8][2][1][1]["isAsync"] and properties[8][2][1][1]["generator"] and properties[-1][0] == "spreadelement"
meta = tree("new.target")
assert meta[1] == {"meta": ("identifier", {"name": "new"}), "property": ("identifier", {"name": "target"})}
assert tree("import.meta")[1]["meta"][1]["name"] == "import"
assert tree("super.x")[2][0][0] == "super" and tree("this")[0] == "thisexpression"
assert tree("(await a) ** 2")[2][0][0] == "parenthesizedexpression"
for text in ("a ?? b as T", "a || b && c as number < d", "a?.b.c!.d?.[e]?.(f)", "x = (a, b)", "f<T>(a)", "new A<T>()",
             "new.target", "(x as T).y", "x as A.B.C", "<T>x", "#x in o", "a **= b", "a?.(b)!", "(await a) ** 2",
             "(-a) ** 2", "f`a${b}`", "x satisfies A & B.C.D"):
    assert value(text) == text, (text, value(text))
assert value("new A") == "new A()" and value("a => a") == "(a) => a"
print("ok")

## TS5PRS-05 · Statements: declarations of variables, conditions, loops, switches and their cases, exceptions and their `catch`, labels and jumps

In [ ]:
declaration = first("let a = 1, [b, , ...c] = d, { e, f: g = 2, ...h } = i;", shapes=True)
assert declaration["declarationKind"] == "let" and kinds(declaration["declarations"]) == ["VariableDeclarator"] * 3
pattern = declaration["declarations"][2]["id"]
assert pattern[0] == "objectpattern" and [p[0] for p in pattern[2]] == ["property", "property", "restelement"]
assert pattern[2][0][1]["shorthand"] and pattern[2][1][2][1][0] == "assignmentpattern"
assert [e[0] for e in declaration["declarations"][1]["id"][2]] == ["identifier", "elision", "restelement"]
defaulted = first("const { a = 1 } = b;", shapes=True)["declarations"][0]["id"][2][0]
assert defaulted[1]["shorthand"] and defaulted[2][1][0] == "assignmentpattern"
assert first("let x!: number;")["declarations"] == [{"kind": "VariableDeclarator", "id": "x: number", "definite": True}]
assert first("var x;")["declarations"] == [{"kind": "VariableDeclarator", "id": "x"}]
assert first("using r = f();")["declarationKind"] == "using"
assert first("await using r = f();")["declarationKind"] == "await using"
assert first("for (using x of y) {}")["left"]["declarationKind"] == "using"
assert first("for (await using x of y) {}")["left"]["declarationKind"] == "await using"
assert first("for await (const x of y) {}")["isAwait"] is True and "isAwait" not in first("for (x of y) {}")
assert first("for (var x = 1 in o) {}")["left"]["declarations"][0]["init"] == "1"
loop = first("for (x in y) z;")
assert (loop["kind"], loop["left"], loop["body"]["kind"]) == ("ForInStatement", "x", "ExpressionStatement")
assert first("for (const [a, b] of c) {}")["left"]["declarations"][0]["id"] == "[a, b]"
assert first("for (;;) {}") == {"kind": "ForStatement", "line": 1, "body": {"kind": "BlockStatement", "line": 1}}
counted = first("for (i = 0; i < 1; i++) {}")
assert (counted["init"], counted["test"], counted["update"]) == ("i = 0", "i < 1", "i++")
assert first("for (let i = 0;;) {}")["init"]["kind"] == "VariableDeclaration"
switch = first("switch (a) {\n    case 1:\n        b;\n        break;\n    default:\n        c;\n}")
assert kinds(switch["cases"]) == ["SwitchCase", "SwitchCase"] and [c.get("test") for c in switch["cases"]] == ["1", None]
assert kinds(switch["cases"][0]["consequent"]) == ["ExpressionStatement", "BreakStatement"]
assert [c["line"] for c in switch["cases"]] == [2, 5] and switch["cases"][1]["consequent"][0]["line"] == 6
assert "cases" not in first("switch (a) { // dropped: before every case\n}")
attempt = first("try { a; } catch (e: unknown) { b; } finally { c; }")
assert attempt["handler"]["kind"] == "CatchClause" and attempt["handler"]["param"] == "e: unknown"
assert kinds(attempt["block"]["body"]) == kinds(attempt["finalizer"]["body"]) == ["ExpressionStatement"]
assert "param" not in first("try {} catch {}")["handler"] and "handler" not in first("try {} finally {}")
assert first("try {} catch ({ message }) {}", shapes=True)["handler"]["param"][0] == "objectpattern"
chain = first("if (a) b; else if (c) d; else e;")
assert chain["alternate"]["kind"] == "IfStatement" and chain["alternate"]["alternate"]["expression"] == "e"
labeled = first("l: for (;;) { break l; continue l; }")
assert labeled["label"] == {"kind": "Identifier", "name": "l"}
assert [s["label"]["name"] for s in labeled["body"]["body"]["body"]] == ["l", "l"]
assert first("throw e;") == {"kind": "ThrowStatement", "line": 1, "argument": "e"}
assert first("function f() { return; }")["body"]["body"] == [{"kind": "ReturnStatement", "line": 1}]
assert kinds(parse("do {} while (a);\nwhile (a) b;\nwith (o) {}\ndebugger;\n;\nbreak;\ncontinue;")) == [
    "DoWhileStatement", "WhileStatement", "WithStatement", "DebuggerStatement", "EmptyStatement", "BreakStatement",
    "ContinueStatement"]
assert first("do\n    a;\nwhile (b);")["body"] == {"kind": "ExpressionStatement", "line": 2, "expression": "a"}
assert first("{ a; }") == {"kind": "BlockStatement", "line": 1, "body": [
    {"kind": "ExpressionStatement", "line": 1, "expression": "a"}]}
print("ok")

## TS5PRS-06 · Functions and classes: signatures, overloads, members and their lines, modifiers, decorators and parameter properties; a class expression's members in its `classBody`

In [ ]:
function = first("export async function* f<T>(this: X, a?: T, b = 1, ...c: T[]): AsyncGenerator<T> {}",
                 shapes=True)["declaration"]
assert function["isAsync"] and function["generator"] and [p[0] for p in function["params"]] == [
    "identifier", "identifier", "assignmentpattern", "restelement"]
assert function["params"][1][1]["optional"] and function["returnType"]["kind"] == "TSTypeAnnotation"
assert first("declare function f(): void;")["kind"] == "TSDeclareFunction"
assert first("declare function f(): void;")["declare"] is True
assert "declare" not in first("function f(a: string): string;") and first("function f(a: string): string;")[
    "kind"] == "TSDeclareFunction"
predicate = first("function g(x: unknown): x is string { return true; }", shapes=True)["returnType"]["typeAnnotation"]
assert predicate[0] == "tstypepredicate" and predicate[2][0] == ("identifier", {"name": "x"}, [])
assert first("function g(x: unknown): asserts x {}", shapes=True)["returnType"]["typeAnnotation"][1]["asserts"]
asserts = first("function g(this: X): asserts this is Y {}", shapes=True)["returnType"]["typeAnnotation"]
assert asserts[1]["asserts"] and asserts[2][0] == ("tsthistype", {}, []) and asserts[1]["typeAnnotation"]
klass = first("""@d @e.f()
abstract class C<T> extends B<T> implements I, J<T> {
    #p = 1;
    static q?: number;
    declare readonly r: string;
    protected override s!: number;
    accessor t = 1;
    abstract u: string;
    abstract accessor w: number;
    [k: string]: unknown;
    constructor(private readonly a: T, public b = 1, @g c?: number) { super(); }
    get v(): number { return 1; }
    set v(x) {}
    static { C.q = 1; }
    m?(): void;
    abstract n(): void;
    'constructor'() {}
    [Symbol.iterator]() {}
    @h async *o() {}
}""", shapes=True)
assert klass["abstract"] and len(klass["decorators"]) == 2 and len(klass["implements"]) == 2
assert klass["superClass"] == ("identifier", {"name": "B"}, []) and klass["superTypeArguments"]["params"] == [
    ("tstypereference", {}, [("identifier", {"name": "T"}, [])])]
members = klass["body"]["body"]
assert kinds(members) == [
    "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "AccessorProperty",
    "TSAbstractPropertyDefinition", "TSAbstractAccessorProperty", "TSIndexSignature", "MethodDefinition",
    "MethodDefinition", "MethodDefinition", "StaticBlock", "MethodDefinition", "TSAbstractMethodDefinition",
    "MethodDefinition", "MethodDefinition", "MethodDefinition"]
assert [m.get("line") for m in members] == [3, 4, 5, 6, 7, 8, 9, None, 11, 12, 13, 14, 15, 16, 17, 18, 19]  # but an
# index signature's, which `TypeScript Types` gives no line
assert members[0]["key"] == ("privateidentifier", {"name": "p"}, []) and members[1]["static"] and members[1]["optional"]
assert members[2]["declare"] and members[2]["readonly"] and members[3]["accessibility"] == "protected"
assert members[3]["override"] and members[3]["definite"]
constructor = members[8]
assert constructor["methodKind"] == "constructor" and constructor["value"][0] == "functionexpression"
parameters = constructor["value"][2][2:5]  # its subterms: its parameters, after its name and type parameters
assert [p[0] for p in parameters] == ["tsparameterproperty", "tsparameterproperty", "identifier"]
assert parameters[0][1] == {"accessibility": "private", "readonly": True} and parameters[1][1] == {
    "accessibility": "public"}
assert parameters[2][1]["optional"] and parameters[2][2][0] == ("identifier", {"name": "g"}, [])  # its decorator's
assert constructor["value"][1]["block"]["body"][0]["expression"][0] == "callexpression"  # `super()`, in its block
assert [m["methodKind"] for m in members[9:11]] == ["get", "set"]
assert kinds(members[11]["body"]) == ["ExpressionStatement"] and members[12]["optional"]
assert members[12]["value"][0] == "tsemptybodyfunctionexpression" and members[13]["value"][0] == (
    "tsemptybodyfunctionexpression")
assert members[14]["methodKind"] == "constructor" and members[15]["computed"]
assert len(members[16]["decorators"]) == 1 and members[16]["value"][1]["isAsync"] and members[16]["value"][1][
    "generator"]
assert first("class D { static constructor() {} }")["body"]["body"][0]["methodKind"] == "method"
heritage = first("class X extends B implements a.b.C, d.E<T> {}")
assert "superTypeArguments" not in heritage and heritage["implements"][0] == {
    "kind": "TSClassImplements", "expression": "a.b.C"}
assert first("interface I extends a.B {}", shapes=True)["extends"][0]["expression"][0] == "memberexpression"
expression = value("(class A extends B { m() { return 1; } })", shapes=True)[2][0]
assert expression[0] == "classexpression" and expression[1]["id"] == ("identifier", {"name": "A"})
assert expression[1]["classBody"]["kind"] == "ClassBody"  # this grammar's class body, in the term's `classBody`
assert expression[1]["classBody"]["body"][0]["value"][1]["block"]["body"][0]["kind"] == "ReturnStatement"
named = value("(function named(a) { return named; })", shapes=True)[2][0]
assert named[0] == "functionexpression" and named[1]["id"] == ("identifier", {"name": "named"})
assert named[1]["block"]["body"][0]["argument"] == ("identifier", {"name": "named"}, [])
assert value("(function* () {})", shapes=True)[2][0][1]["generator"]
assert first("export default class {}")["declaration"] == {"kind": "ClassDeclaration", "line": 1,
                                                           "body": {"kind": "ClassBody"}}
assert "id" not in first("export default function () {}")["declaration"]
assert first("export default function* g() {}")["declaration"]["generator"]
assert first("export default class C {}")["declaration"]["id"] == {"kind": "Identifier", "name": "C"}
assert len(first("@d export class E {}")["declaration"]["decorators"]) == 1
assert len(first("@d export default class {}")["declaration"]["decorators"]) == 1
assert len(first("@d export default class C {}")["declaration"]["decorators"]) == 1
print("ok")

## TS5PRS-07 · Modules: imports, exports, attributes, import equals, namespaces, ambient modules and `declare global`; TypeScript's declarations: aliases, interfaces and enums

In [ ]:
imported = first("import d, { a as b, type c, 'e' as f } from 'm' with { type: 'json' };")
assert kinds(imported["specifiers"]) == ["ImportDefaultSpecifier", "ImportSpecifier", "ImportSpecifier", "ImportSpecifier"]
assert imported["specifiers"][2]["importKind"] == "type" and imported["specifiers"][3]["imported"] == "'e'"
assert imported["attributes"] == [{"kind": "ImportAttribute", "key": "type", "value": {
    "kind": "Literal", "value": "json", "text": "'json'", "quote": "'"}}]
assert imported["source"] == {"kind": "Literal", "value": "m", "text": "'m'", "quote": "'"}
assert first("import type { T } from 'm';")["importKind"] == "type" and imported["importKind"] == "value"
assert first("import * as ns from 'm';")["specifiers"] == [
    {"kind": "ImportNamespaceSpecifier", "local": {"kind": "Identifier", "name": "ns"}}]
assert "specifiers" not in first("import 'side';")
required = first("import x = require('m');")
assert required["moduleReference"]["kind"] == "TSExternalModuleReference" and required["moduleReference"][
    "expression"]["text"] == "'m'"
assert first("import type x = require('m');")["importKind"] == "type"
assert first("import y = A.B.C;")["moduleReference"]["kind"] == "TSQualifiedName"
assert first("import y = A;")["moduleReference"] == {"kind": "Identifier", "name": "A"}
assert first("export * as ns from 'm';")["exported"] == "ns" and "exported" not in first("export * from 'm';")
assert first("import a from 'm' with { 'type': 'json' };", shapes=True)["attributes"][0]["key"][0] == "literal"
assert first("export * as 'n s' from 'm';", shapes=True)["exported"][0] == "literal"
named = first("export { a as b, type c, 'd' } from 'm';", shapes=True)
assert [s["exportKind"] for s in named["specifiers"]] == ["value", "type", "value"] and named["source"]["value"] == "m"
assert named["specifiers"][2]["local"][0] == named["specifiers"][2]["exported"][0] == "literal"
assert first("export { a as 'b c' };")["specifiers"][0]["exported"] == "'b c'"
assert first("export type { T };")["exportKind"] == "type"
assert first("export interface I {}")["exportKind"] == "type" and first("export type A = B;")["exportKind"] == "type"
assert first("export declare const x: number;")["exportKind"] == "type"
assert first("export const y = 1;")["exportKind"] == "value"
assert first("export = x;") == {"kind": "TSExportAssignment", "line": 1, "expression": "x"}
assert first("export as namespace N;")["id"] == {"kind": "Identifier", "name": "N"}
assert first("export default a + b;", shapes=True)["declaration"][0] == "binaryexpression"
assert first("export default (function() {});", shapes=True)["declaration"][0] == "parenthesizedexpression"
namespace = first("namespace A.B { export const x = 1; }")
assert namespace["id"]["kind"] == "TSQualifiedName" and namespace["moduleKind"] == "namespace"
assert kinds(namespace["body"]["body"]) == ["ExportNamedDeclaration"]
assert first("module M {}")["moduleKind"] == "module"
assert first("x = 1;\nnamespace N {}")["kind"] == "ExpressionStatement"
ambient = parse('declare module "m" { export const v: number; }\ndeclare module "short";\nx;')
assert kinds(ambient) == ["TSModuleDeclaration", "TSModuleDeclaration", "ExpressionStatement"]
assert ambient[0]["declare"] and ambient[0]["id"]["kind"] == "Literal" and "body" not in ambient[1]
assert first("declare global { interface W {} }")["moduleKind"] == "global"
in_module = first('declare module "m" { global { interface W {} } }')["body"]["body"][0]
assert in_module["moduleKind"] == "global" and "declare" not in in_module and kinds(in_module["body"]["body"]) == [
    "TSInterfaceDeclaration"]
alias = first("type A<T> = T[];")
assert alias["typeParameters"]["params"][0]["name"] == {"kind": "Identifier", "name": "T"} and alias["typeAnnotation"] == (
    "T[]")
assert first("type U<S extends string> = intrinsic;", shapes=True)["typeAnnotation"] == ("tsintrinsickeyword", {}, [])
assert first("declare type A = B;")["declare"]
interface = first("interface I<T> extends A.B<T>, C { a: T; }")
assert interface["extends"][0]["expression"] == "A.B" and interface["extends"][0]["typeArguments"]["params"] == ["T"]
assert kinds(interface["body"]["body"]) == ["TSPropertySignature"]
enum = first("declare const enum E { A, B = A + 1, 'C-d', [k] = 2 }")
assert enum["const"] and enum["declare"] and kinds(enum["body"]["members"]) == ["TSEnumMember"] * 4
assert [m.get("computed", False) for m in enum["body"]["members"]] == [False, False, False, True]
assert [m.get("initializer") for m in enum["body"]["members"]] == [None, "A + 1", None, "2"]
assert "members" not in first("enum E {}")["body"] and first("enum E { A }")["body"]["members"][0]["line"] == 1
print("ok")

## TS5PRS-08 · Types (`TypeScript Types`), with TypeScript's grouping where tree-sitter-typescript's differs: `readonly`, and unions and intersections flattened

In [ ]:
def type_of(text, shapes=True):
    return first(f"type X = {text};", shapes)["typeAnnotation"]


keywords = type_of("any | unknown | number | bigint | boolean | string | symbol | object | never | void | undefined | "
                   "null | this")
assert [t[0] for t in keywords[2]] == [
    "tsanykeyword", "tsunknownkeyword", "tsnumberkeyword", "tsbigintkeyword", "tsbooleankeyword", "tsstringkeyword",
    "tssymbolkeyword", "tsobjectkeyword", "tsneverkeyword", "tsvoidkeyword", "tsundefinedkeyword", "tsnullkeyword",
    "tsthistype"]
assert type_of("unique symbol") == ("tstypeoperator", {"operator": "unique"}, [("tssymbolkeyword", {}, [])])
readonly = type_of("readonly A[] | B")  # `readonly` binds tighter than `|`
assert readonly[0] == "tsuniontype" and readonly[2][0][1] == {"operator": "readonly"}
assert [t[0] for t in type_of("readonly A[] | B | C")[2]] == ["tstypeoperator", "tstypereference", "tstypereference"]
assert type_of("readonly A[] & B")[0] == "tsintersectiontype"
assert [t[0] for t in type_of("A | readonly B[] | C")[2]] == ["tstypereference", "tstypeoperator", "tstypereference"]
assert [t[0] for t in type_of("A & (B | C) & D")[2]] == ["tstypereference", "tsparenthesizedtype", "tstypereference"]
assert type_of("keyof T")[1] == {"operator": "keyof"}
assert [q[1]["raw"] for q in type_of("`a${B}c`")[1]["quasis"]] == ["a", "c"]
assert type_of("-1")[2][0][0] == "unaryexpression" and type_of("'a'")[2][0][1]["text"] == "'a'"
assert type_of("true")[2][0][1]["value"] is True
assert type_of("null")[0] == "tsnullkeyword" and type_of("undefined")[0] == "tsundefinedkeyword"
tuple_type = type_of("[a: string, b?: number, ...c: boolean[]]")[2]
assert [t[0] for t in tuple_type] == ["tsnamedtuplemember", "tsnamedtuplemember", "tsresttype"]
assert tuple_type[1][1]["optional"] and tuple_type[2][2][0][0] == "tsnamedtuplemember"
assert [t[0] for t in type_of("[string?, ...number[]]")[2]] == ["tsoptionaltype", "tsresttype"]
assert type_of("abstract new () => object")[1]["abstract"]
function = type_of("<T>(a: T) => a is string")  # its subterms: its parameters, then its return type's
assert function[0] == "tsfunctiontype" and function[1]["returnType"] == ("tstypeannotation", {})
assert function[2][2][1]["name"] == "a" and function[2][-1][0] == "tstypepredicate"
assert type_of("(a: unknown) => asserts a")[2][-1] == ("tstypepredicate", {"asserts": True}, [
    ("identifier", {"name": "a"}, [])])
mapped = type_of("{ -readonly [K in keyof T as `get${K & string}`]+?: T[K] }")
assert mapped[0] == "tsmappedtype" and (mapped[1]["readonly"], mapped[1]["optional"]) == ("-readonly", "+?")
assert mapped[1]["key"] == ("identifier", {"name": "K"})  # its constraint its `value`, its type its `body`, then
assert [t[0] for t in mapped[2]] == ["tstypeoperator", "tsindexedaccesstype", "tstemplateliteraltype"]  # its name's
assert (type_of("{ readonly [K in T]-?: K }")[1]["readonly"], type_of("{ [K in T]?: K }")[1]["optional"]) == (
    "readonly", "?")
assert type_of("{ +readonly [K in T]: K }")[1]["readonly"] == "+readonly"
assert "optional" not in type_of("{ [K in T]: K }")[1]
conditional = type_of("T extends (infer U extends string)[] ? U : never")
inferred = conditional[2][1][2][0][2][0]
assert inferred[:2] == ("tsinfertype", {"typeParameter": ("tstypeparameter", {"name": ("identifier", {"name": "U"})})})
assert inferred[2] == [("tsstringkeyword", {}, []), None]  # its constraint
assert type_of("T extends (infer U)[] ? U : never")[2][1][2][0][2][0][2] == [None, None]
assert type_of("typeof a.b<T>")[1]["typeArguments"][0] == "tstypeparameterinstantiation"
assert type_of("typeof import('m')")[2][0][0] == "tsimporttype"
imported = type_of("import('m', { with: { 'resolution-mode': 'import' } }).A.B<T>")
assert imported[0] == "tsimporttype" and [t[0] for t in imported[2][:3]] == ["literal", "objectexpression",
                                                                             "tsqualifiedname"]
assert type_of("import('m')", False) == "import('m')" and type_of("import('m').X")[2][2] == (
    "identifier", {"name": "X"}, [])
queried = first("let a: typeof import('m').x;", shapes=True)["declarations"][0]["id"][2][0]  # its annotation's type
assert queried[0] == "tstypequery" and queried[2][0][0] == "tsimporttype"
assert type_of("A['b'][C]")[0] == "tsindexedaccesstype"
members = type_of("{ a: string; readonly b?: number; c(): void; get d(): number; set d(v); (x): y; new (x): y; "
                  "[k: string]: unknown; static e: string }")[1]["members"]
assert [m[0] for m in members] == [
    "tspropertysignature", "tspropertysignature", "tsmethodsignature", "tsmethodsignature", "tsmethodsignature",
    "tscallsignaturedeclaration", "tsconstructsignaturedeclaration", "tsindexsignature", "tspropertysignature"]
assert members[1][1]["readonly"] and members[3][1]["methodKind"] == "get" and members[8][1]["static"]
assert type_of("(A)")[0] == "tsparenthesizedtype" and type_of("A.B.C")[2][0][0] == "tsqualifiedname"
assert type_of("bigint")[0] == "tsbigintkeyword" and type_of("typeof this")[2][0][0] == "thisexpression"
assert type_of("new <T>() => T")[1]["typeParameters"][0] == "tstypeparameterdeclaration"
assert ["typeAnnotation" in m[1] for m in type_of("{ a; b? }")[1]["members"]] == [False, False]
assert type_of("readonly string[]")[0] == "tstypeoperator"
for text in ("readonly A[] | B", "keyof T", "[a: string, b?: number, ...c: boolean[]]", "abstract new () => object",
             "{ -readonly [K in keyof T as `get${K & string}`]+?: T[K] }",
             "T extends (infer U extends string)[] ? U : never", "typeof a.b<T>", "import('m').A.B<T>", "A['b'][C]",
             "(A | B)[]", "unique symbol", "`a${B}c`", "-1", "(() => void) | null", "<T>(a: T) => a is string",
             "typeof this", "new <T>() => T", "readonly string[]"):
    assert type_of(text, False) == text, (text, type_of(text, False))
assert first("class X extends B implements a.b.C {}")["implements"][0]["expression"] == "a.b.C"
print("ok")

## TS5PRS-09 · JSX, with the `tsx` grammar: elements, fragments, attributes, children and the text between them; `<T>x` a type assertion only in another file

In [ ]:
element = tsx("<a.b x='1' y={2} {...z} c:d=\"e\" f>t &amp; u{v}{/* c */}{...w}<g /><>h</></a.b>", shapes=True)
opening = element[1]["openingElement"][1]
assert opening["name"] == ("jsxmemberexpression", {"object": ("jsxidentifier", {"name": "a"}),
                                                   "property": ("jsxidentifier", {"name": "b"})})
assert [a[0] for a in opening["attributes"]] == [
    "jsxattribute", "jsxattribute", "jsxspreadattribute", "jsxattribute", "jsxattribute"]
assert opening["attributes"][0][1]["value"] == ("literal", {"value": "1", "text": "'1'", "quote": "'"})
assert opening["attributes"][3][1]["name"][0] == "jsxnamespacedname" and "value" not in opening["attributes"][4][1]
assert [c[0] for c in element[1]["children"]] == ["jsxtext", "jsxexpressioncontainer", "jsxexpressioncontainer",
                                                  "jsxspreadchild", "jsxelement", "jsxfragment"]
assert element[1]["children"][0] == ("jsxtext", {"raw": "t &amp; u"})
assert ("jsxemptyexpression", {}, []) in element[2]  # `{/* c */}`: an empty expression
assert element[1]["closingElement"][1]["name"][0] == "jsxmemberexpression"
assert tsx("<a x='\\n' />", shapes=True)[1]["openingElement"][1]["attributes"][0][1]["value"][1]["value"] == "\\n"
assert tsx("<a x=<b /> />", shapes=True)[1]["openingElement"][1]["attributes"][0][1]["value"][0] == "jsxelement"
assert tsx("<C<T> />", shapes=True)[1]["openingElement"][1]["typeArguments"][0] == "tstypeparameterinstantiation"
assert tsx("<div>\n    a\n</div>", shapes=True)[1]["children"] == [("jsxtext", {"raw": "\n    a\n"})]
assert tsx("<div>  </div>", shapes=True)[1]["children"] == [("jsxtext", {"raw": "  "})]
assert "children" not in tsx("<div></div>", shapes=True)[1]
assert tsx("<T,>(a: T) => a", shapes=True)[1]["typeParameters"][0] == "tstypeparameterdeclaration"
for text in ("<a.b x='1' y={2} {...z} f>t &amp; u{v}{...w}<g /><>h</></a.b>", "<C<T> />", "<div>{}</div>",
             "<T,>(a: T) => a"):
    assert tsx(text) == text, (text, tsx(text))
assert value("<T>x", shapes=True)[0] == "tstypeassertion"  # in a `.ts` file, a type assertion
assert first("<T>x;", suffix=".tsx") == {"kind": "Error", "line": 1, "text": "<T>x;"}  # in a `.tsx` file, no JSX
print("ok")

## TS5PRS-10 · Comments are statements where statements, class members and enum members are listed, on their own lines or trailing; `TypeScript Types`' own among an interface's members; elsewhere they are dropped

In [ ]:
listed = parse("""// one
a; /* two */
class C { // three
    // four
    m() {} // five
}
interface I { /* six */ a: string; }
enum E { A, // seven
}
switch (x) { case 1: // eight
}
f(/* dropped */ a);
""")
assert kinds(listed) == ["Comment", "ExpressionStatement", "Comment", "ClassDeclaration", "TSInterfaceDeclaration",
                         "TSEnumDeclaration", "SwitchStatement", "ExpressionStatement"]
assert listed[0] == {"kind": "Comment", "line": 1, "text": " one"}
assert listed[2] == {"kind": "Comment", "line": 2, "text": " two ", "block": True, "trailing": True}
assert [(c["text"], c.get("trailing", False), c["line"]) for c in listed[3]["body"]["body"] if c["kind"] == "Comment"] == [
    (" three", True, 3), (" four", False, 4), (" five", True, 5)]
assert listed[4]["body"]["body"][0] == {"kind": "Comment", "block": True, "text": " six ", "trailing": True}  # no line
assert listed[5]["body"]["members"][1] == {"kind": "Comment", "line": 8, "text": " seven", "trailing": True}
assert listed[6]["cases"][0]["consequent"] == [{"kind": "Comment", "line": 10, "text": " eight", "trailing": True}]
assert listed[7]["expression"] == "f(a)"
nested = first("function f() {\n    // inside\n    if (a) {\n    } // after\n    b;\n}")
assert kinds(nested["body"]["body"]) == ["Comment", "IfStatement", "Comment", "ExpressionStatement"]
assert nested["body"]["body"][2]["trailing"]
assert type_of("{\n    // eight\n    a: string;\n}")[1]["members"][0][1] == {"text": " eight"}
print("ok")

## TS5PRS-11 · What tree-sitter-typescript 0.23 cannot parse, which the pre-pass puts back

In [ ]:
klass = first("class C { accessor a = 1; static accessor b: number; accessor() {} accessor = 2; accessor }")
assert kinds(klass["body"]["body"]) == ["AccessorProperty", "AccessorProperty", "MethodDefinition",
                                        "PropertyDefinition", "PropertyDefinition"]
assert [m["key"] for m in klass["body"]["body"]] == ["a", "b", "accessor", "accessor", "accessor"]
assert klass["body"]["body"][1]["static"]
parameters = first("type T<in A, out B, in out C, const in D> = A;")["typeParameters"]["params"]
assert [(p.get("isIn", False), p.get("isOut", False)) for p in parameters] == [
    (True, False), (False, True), (True, True), (True, False)]
assert parameters[3]["const"] and first("function f<in>() {}")["typeParameters"]["params"][0]["name"]["name"] == "in"
assert first("import defer * as m from 'm';")["phase"] == "defer" and "phase" not in first("import { defer } from 'm';")
assert first("import defer { a } from 'm';")["phase"] == "defer"
assert first("export type * from 'm';")["exportKind"] == "type"
assert first("export type * as ns from 'm';")["exportKind"] == "type"
abstract = first("class C { abstract: number; readonly abstract?: string; abstract() {} }")["body"]["body"]
assert [m["key"] for m in abstract] == ["abstract", "abstract", "abstract"]
assert value("using + 1", shapes=True)[2][0] == ("identifier", {"name": "using"}, [])
assert value("x = using") == "x = using" and value("using in o") == "using in o"
annotated = first("let x: import('m').A<B>;", shapes=True)["declarations"][0]["id"][2][0]  # its annotation's type
assert annotated[0] == "tsimporttype" and annotated[1]["typeArguments"][0] == "tstypeparameterinstantiation"
assert [t and t[0] for t in annotated[2]] == ["literal", None, "identifier", "tstypereference"]
spanning = parse("let x: import(\n'm').A<B>;\n// after\ny;")  # the import spans lines
assert spanning[0]["declarations"][0]["id"] == "x: import('m').A<B>"
assert spanning[1] == {"kind": "Comment", "line": 3, "text": " after"}  # lines are those written: not trailing
called = value("a<import('m').B>()", shapes=True)
assert called[2][1][0] == "tsimporttype"  # its type argument
print("ok")

## TS5PRS-12 · Invalid syntax is kept, never raised: an `Error` where a statement, a case, a class member or an enum member stands, an `error` or `missing` term within an expression, a type or a pattern, each as written; only text that is no text raises

In [ ]:
# Where a statement stands: an `Error`, its text as written, its line, and the statements recovered within it
mixed = parse("a = 1;\nif (a {}\nb = 3;\n")
assert kinds(mixed) == ["ExpressionStatement", "Error", "ExpressionStatement"]
assert mixed[1] == {"kind": "Error", "line": 2, "text": "if (a {}"}  # a `MISSING` `)`, which no term holds
assert parse("a = 1;\nlet = ;") == [{"kind": "ExpressionStatement", "line": 1, "expression": "a = 1"},
                                     {"kind": "Error", "line": 2, "text": "let = ;"}]  # what tree-sitter skipped
assert first("let x: import('m'") == {"kind": "Error", "line": 1, "text": "let x: import('m'"}
recovered = first("if (a {\n    b;\n    { c; }\n}")
assert recovered["text"] == "if (a {\n    b;\n    { c; }\n}" and kinds(recovered["body"]) == [
    "ExpressionStatement", "BlockStatement"]
assert recovered["body"][0]["line"] == 2
nested = first("function f() {\n    let = ;\n    y;\n}")
assert kinds(nested["body"]["body"]) == ["Error", "ExpressionStatement"]
assert nested["body"]["body"][0] == {"kind": "Error", "line": 2, "text": "let = ;"}
assert first("try {} catch (e +) {}")["kind"] == "Error"  # a clause's: its statement is invalid whole
assert first("interface I { a: ; }")["kind"] == "Error"  # an interface member's too
assert first("switch (a) { case 1: b; case 2 3: c; }")["cases"][1] == {
    "kind": "Error", "line": 1, "text": "case 2 3: c;", "body": [{"kind": "ExpressionStatement", "line": 1,
                                                                   "expression": "c"}]}
assert first("enum E { A = , B }")["body"]["members"] == [{"kind": "Error", "line": 1, "text": "A = , B"}]
member = first("class C { accessor a = 1; m( }")["body"]["body"]
assert kinds(member) == ["AccessorProperty", "Error"] and member[1] == {"kind": "Error", "line": 1, "text": "m("}
for text in ("[K in T]: K", "@d static {}", "@d [k: string]: unknown"):  # constructs no member holds
    assert first(f"class C {{\n    {text}\n}}")["body"]["body"] == [{"kind": "Error", "line": 2, "text": text}]
assert parse("x;\n<div />;") == [{"kind": "ExpressionStatement", "line": 1, "expression": "x"},
                                   {"kind": "Error", "line": 2, "text": "<div />;"}]  # JSX, in a `.ts` file
assert parse("accessor;\nlet = ;")[1]["text"] == "let = ;"  # around what the pre-pass rewrote
assert first("export type * from ;")["text"] == "export type * from ;"  # as written, not as the pre-pass rewrote it
# Within an expression: an `error` term, the innermost expression that holds it, with the terms recovered within it
assert value("x = = 2", shapes=True) == ("error", {"text": "x = = 2"}, [
    ("identifier", {"name": "x"}, []), ("literal", {"value": 2.0, "text": "2", "base": 10}, [])])
assert value("f(a,,b)", shapes=True)[1:] == ({"text": "f(a,,b)"}, [("identifier", {"name": n}, []) for n in "fab"])
assert value("new A(,)", shapes=True)[:2] == ("error", {"text": "new A(,)"})  # a hole among arguments
assert value("x = (a: T)", shapes=True)[2][1][:2] == ("error", {"text": "(a: T)"})  # Flow's annotation
assert value("x?.y = 1", shapes=True)[2][0] == ("error", {"text": "x?.y"}, [("identifier", {"name": "x"}, [])])
assert value("x = '\\u{110000}'", shapes=True)[2][1] == ("error", {"text": "'\\u{110000}'"}, [])  # no character
assert value("'\\u{FFFFFFFFFFFF}'", shapes=True)[0] == "error"
assert value("x = (a, 😀 b)", shapes=True)[2][1][1] == {"text": "(a, 😀 b)"}  # as written, beyond the first plane
arrow = value("x = () => { a = = 1; b; }", shapes=True)[2][1][1]["block"]["body"]  # the block's own, kept within
assert arrow[0]["expression"][:2] == ("error", {"text": "a = = 1"}) and arrow[1]["expression"][0] == "identifier"
# Within a type: `TypeScript Types`' `error` term
assert first("type X = ?string;", shapes=True)["typeAnnotation"] == ("error", {"text": "?string"}, [])
assert first("type X = { [K in T]: K; a: string };", shapes=True)["typeAnnotation"][0] == "error"
# `MISSING` where an expression stands: a `missing` term, the kind tree-sitter expected
assert value("(1 + )", shapes=True)[2][0][2][1] == ("missing", {"kind": "identifier"}, [])
assert first("switch (a) { case 1 +: b; }", shapes=True)["cases"][0]["test"][2][1][0] == "missing"
# Kept in the store and in JSON as any statement and term are
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
    Parse.parse("a = 1;\nif (a {}\nx = = 2;\nz = (1 + );\n", "invalid.ts", STORE))))
fresh = Programs.store()
(again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
with fresh.transaction():
    nodes = [Terms.kind_of(n)[1] for n in Objects.properties(program(fresh, "invalid.ts"))["body"].values]
    assert [node.as_schema().name for node in nodes] == ["Programs.TypeScript5.ExpressionStatement",
                                                          "Programs.TypeScript5.Error",
                                                          "Programs.TypeScript5.ExpressionStatement",
                                                          "Programs.TypeScript5.ExpressionStatement"]
    assert [Objects.properties(node)["line"] for node in nodes] == [1, 2, 3, 4]
    assert Objects.properties(nodes[1]) == {"line": 2, "text": "if (a {}"}
    assert Terms.kind_of(Objects.properties(nodes[2])["expression"])[0] == "error"
# Only text that is no text raises, before any transaction
fails("x = '\ud800';", "not text: a lone surrogate", 1, 6)
fails("x = 1;\n  \udfff", "not text: a lone surrogate", 2, 3)
with STORE.transaction():
    assert "failing.ts" not in [e.path.get() for e in STORE.singleton(Model.WORKSPACE).files]  # nothing written
print("ok")

## TS5PRS-13 · Where tree-sitter-typescript departs from TypeScript 5: each place `Parse`'s docstring enumerates, by its number, with the tree read and the text printed back

In [ ]:
from mbse.Programs.TypeScript5 import Print


def printed(text):
    """The text the program `text` parses to prints as."""
    path = next(PATHS) + ".ts"
    STORE.transaction(Parse.parse(text, path))
    return Print.print_module(STORE, path)


def same(text):
    """Whether `text`, a program in the printer's layout, prints back as written."""
    return printed(text) == text


# What tree-sitter-typescript cannot parse, which the pre-pass rewrites
# 1. `accessor` after another modifier
members = first("class C {\n    static accessor a = 1;\n    override accessor b = 2;\n}")["body"]["body"]
assert kinds(members) == ["AccessorProperty", "AccessorProperty"] and members[0]["static"] and members[1]["override"]
assert first("abstract class C {\n    abstract accessor a: number;\n}")["body"]["body"][0]["kind"] == (
    "TSAbstractAccessorProperty")
assert same("class C {\n    static accessor a = 1;\n    override accessor b = 2;\n}\n")
# 2. Variance annotations
parameters = first("interface I<in A, out B, in out C> {}")["typeParameters"]["params"]
assert [(p.get("isIn", False), p.get("isOut", False)) for p in parameters] == [(True, False), (False, True), (True, True)]
assert same("class C<in out T> {}\n") and same("type T<const in D> = D;\n")
# 3. `import defer`
assert first("import defer * as m from './m';")["phase"] == "defer" and same('import defer * as m from "./m";\n')
# 4. `import.defer(...)`
deferred = value("import.defer('./m', { with: {} })", shapes=True)
assert deferred[:2] == ("importexpression", {"phase": "defer"}) and [t[0] for t in deferred[2]] == [
    "literal", "objectexpression"]
assert same('const m = await import.defer("./m");\n')
# 5. `export type *`
assert first("export type * as ns from './m';")["exportKind"] == "type" and same('export type * as ns from "./m";\n')
# 6. Attributes on `export ... from`
for text in ('export {a} from "./m" with { type: "json" };\n', 'export * from "./m" with { type: "json" };\n',
             'export type * as ns from "./m" with { "resolution-mode": "import" };\n'):
    assert first(text)["attributes"][0]["kind"] == "ImportAttribute" and same(text), text
assert first('export * from "./m" with { type: "json" };')["attributes"] == [{"kind": "ImportAttribute", "key": "type",
    "value": {"kind": "Literal", "value": "json", "text": '"json"', "quote": '"'}}]
assert first("export {a} from './m' with { type: 'json' };")["kind"] == "ExportNamedDeclaration"
assert printed('export {a} from "./m" assert { type: "json" };') == 'export {a} from "./m" with { type: "json" };\n'
assert "attributes" not in first('export {a} from "./m";')
# 7. `using` declarations in `for (... of ...)` and with a type
assert first("for (await using x of y) {}")["left"]["declarationKind"] == "await using"
annotated = first("await using a: T = f(), b: U = g();")
assert annotated["declarationKind"] == "await using" and [d["id"] for d in annotated["declarations"]] == ["a: T", "b: U"]
assert first("for (using a: T = f(); ; ) {}")["init"]["declarationKind"] == "using"
assert first("switch (x) {\n    case 1:\n        using a: T = f();\n}")["cases"][0]["consequent"][0][
    "declarationKind"] == "using"
assert first("export using a: T = f();")["declaration"]["declarationKind"] == "using"
assert same("using a: T = f();\n") and same("for (await using a: T = f();;) {}\n")
# 8. A class's static index signature
signatures = first("class C {\n    static [k: string]: number;\n    static readonly [k: number]: string;\n}")["body"][
    "body"]
assert kinds(signatures) == ["TSIndexSignature"] * 2 and signatures[0]["static"] and signatures[1]["readonly"]
assert same("class C {\n    static readonly [k: number]: string;\n}\n")
# 9. A decorator's parenthesized expression
klass = first("@(a[0])\nclass C {\n    @(b + c) m(@(d.e[1]) x) {}\n}", shapes=True)
assert klass["decorators"][0]["expression"] == ("parenthesizedexpression", {}, [("memberexpression", {"computed": True}, [
    ("identifier", {"name": "a"}, []), ("literal", {"value": 0.0, "text": "0", "base": 10}, [])])])
assert klass["body"]["body"][0]["decorators"][0]["expression"][2][0][0] == "binaryexpression"
assert same("@(a[0])\nclass C {\n    @(b + c)\n    m(@(d.e[1]) x) {}\n}\n")
# 10. `import("m")` as a type with type arguments (TS5PRS-11 has more)
assert same('let x: import("m").A<B>;\n')
# 11. Type arguments before an optional call
called = value("f<T>?.(a)", shapes=True)
assert called[0] == "chainexpression" and called[2][0][:2] == ("callexpression", {"optional": True})
assert called[2][0][2][0][0] == "tsinstantiationexpression"
assert same("x = a.b<T, U>?.(c);\n")
# 12. Names it reads as keywords
assert value("abstract.x") == "abstract.x" and value("abstract = 1") == "abstract = 1" and value("abstract") == "abstract"
assert first("label: abstract;")["body"]["expression"] == "abstract"
assert [m["key"] for m in first("class C {\n    abstract = 1;\n    abstract!: number;\n}")["body"]["body"]] == [
    "abstract", "abstract"]
assert first("let satisfies = 1;")["declarations"][0]["id"] == "satisfies"
assert [d["id"] for d in first("let as = 1, satisfies;")["declarations"]] == ["as", "satisfies"]
assert first("export { type };")["specifiers"] == [{"kind": "ExportSpecifier", "exportKind": "value", "local": "type",
                                                   "exported": "type"}]
assert [(s["local"], s["exported"]) for s in first("export { type as x, type as as };")["specifiers"]] == [
    ("type", "x"), ("type", "as")]
assert [(s["importKind"], s["imported"], s["local"]["name"]) for s in first(
    "import { type as, type as as x } from 'm';")["specifiers"]] == [("type", "as", "as"), ("type", "as", "x")]
assert first("import d, { type as } from 'm';")["specifiers"][1]["imported"] == "as"
assert value("x = undefined => 1", shapes=True)[2][1][2][2] == ("identifier", {"name": "undefined"}, [])  # its parameter
assert first("let y: any.T;")["declarations"][0]["id"] == "y: any.T" and same("let y: string.T<number>;\n")
assert same("export {type as x};\n") and same("let satisfies = 1;\n")
assert first("export type { type };")["specifiers"][0]["local"] == "type"
assert first("class C {\n    static [k]() {}\n    x = using a;\n}")["body"]["body"][0]["static"]  # no signature
assert parse("f(using x);\nlet = ;")[0]["expression"] == "f(using x)"  # no declaration where no statement starts
# 13. A mapped type without a type
mapped = first("type M = { -readonly [K in T]-? };", shapes=True)["typeAnnotation"]
assert mapped[:2] == ("tsmappedtype", {"key": ("identifier", {"name": "K"}), "readonly": "-readonly", "optional": "-?"})
assert mapped[2] == [("tstypereference", {}, [("identifier", {"name": "T"}, [])]), None, None]  # no type
assert first("type M = { [K in T]; };")["typeAnnotation"] == "{ [K in T] }" and same("type M = { [K in T]? };\n")
assert first("type M = { [K in T]: K };", shapes=True)["typeAnnotation"][2][1] is not None
# 14. `keyof readonly T[]`
keyed = first("type K = keyof readonly T[] | U;", shapes=True)["typeAnnotation"]
assert keyed[0] == "tsuniontype" and keyed[2][0][2][0][:2] == ("tstypeoperator", {"operator": "readonly"})
assert same("type K = keyof readonly T[];\n")
# 15. An anonymous default function's signature
overloads = parse("export default function (a: number): void;\nexport default function (a: any) {}")
assert [(s["declaration"]["kind"], "id" in s["declaration"]) for s in overloads] == [
    ("TSDeclareFunction", False), ("FunctionDeclaration", False)]
assert printed("export default function(a: number): void;") == "export default function(a: number): void;\n"
# 16. An escape of no character in a tagged template (40 has more)
assert value("x = tag`\\unicode${a}\\xg`", shapes=True)[2][1][1]["quasi"][1]["quasis"] == [
    ("templateelement", {"raw": "\\unicode"}), ("templateelement", {"raw": "\\xg", "tail": True})]
assert parse("x;\n\\")[1]["kind"] == "Error"  # a `\` that ends the text escapes nothing
# What tree-sitter-typescript misreads, which the pre-pass rewrites
# 17. A tagged template's type arguments
tagged = value("f<T>`x`", shapes=True)
assert tagged[0] == "taggedtemplateexpression" and tagged[1]["typeArguments"][0] == "tstypeparameterinstantiation"
assert value("a.b<T, U[]>`x${y}`") == "a.b<T, U[]>`x${y}`" and value("f<A<B>>`x`") == "f<A<B>>`x`"
assert value("f<A>>`x`", shapes=True)[0] == "binaryexpression"  # no `<` opens them: a comparison
assert value("f<{ a: 1 }>`x`", shapes=True)[0] == "taggedtemplateexpression" and value("f<(a: T) => U>`x`") == (
    "f<(a: T) => U>`x`")
assert parse("x = a < b;\ny = c > `x`;")[1]["expression"] == "y = c > `x`"  # nor across a statement's end
assert parse("if (a < b) {}\nc > `x`;")[1]["expression"] == "c > `x`"  # nor from within brackets
# 18. `using`, `abstract` and `accessor` a line ends; `using [a] = b`
assert parse("using\nx = 1;") == [{"kind": "ExpressionStatement", "line": 1, "expression": "using"},
                                   {"kind": "ExpressionStatement", "line": 2, "expression": "x = 1"}]
assert kinds(parse("abstract\nclass C {}")) == ["ExpressionStatement", "ClassDeclaration"]
assert "abstract" not in parse("abstract\nclass C {}")[1]
assert first("type T = abstract\nnew () => X;")["typeAnnotation"] == "abstract new () => X"
split = first("class C {\n    accessor\n    x = 1;\n    abstract\n    m() {}\n}")["body"]["body"]
assert [(m["kind"], m["key"]) for m in split] == [("PropertyDefinition", "accessor"), ("PropertyDefinition", "x"),
                                                 ("PropertyDefinition", "abstract"), ("MethodDefinition", "m")]
assert value("using [a] = f()", shapes=True)[2][0] == ("memberexpression", {"computed": True}, [
    ("identifier", {"name": "using"}, []), ("identifier", {"name": "a"}, [])])
assert printed("using\nx = 1;") == "using;\nx = 1;\n" and printed("using [a] = f();") == "using[a] = f();\n"
# 19. Line terminators it reads as none
assert parse("a\rb;") == [{"kind": "ExpressionStatement", "line": 1, "expression": "a"},
                          {"kind": "ExpressionStatement", "line": 1, "expression": "b"}]  # lines count `\n`s
assert kinds(parse("a b;\nc d;")) == ["ExpressionStatement"] * 4
assert parse("function f() {\n    return x;\n}")[0]["body"]["body"][0] == {"kind": "ReturnStatement", "line": 2}
assert parse("// c x;")[1]["expression"] == "x" and value("x = ' '") == "x = ' '"
assert parse("#!/usr/bin/env node x;") == [{"kind": "ExpressionStatement", "line": 1, "expression": "x"}]
assert printed("#!/usr/bin/env node\r\nx;\r\n") == "#!/usr/bin/env node\nx;\n"
# 20. A block comment that is the only line end between two tokens
returned = first("function f() {\n    return /* a\n    b */ x;\n}")["body"]["body"]
assert returned == [{"kind": "ReturnStatement", "line": 2},
                    {"kind": "Comment", "line": 2, "text": " a\n    b ", "block": True, "trailing": True},
                    {"kind": "ExpressionStatement", "line": 3, "expression": "x"}]
assert parse("/*\n*/ a /* */ ++b;")[1:] == [
    {"kind": "ExpressionStatement", "line": 2, "expression": "a"},
    {"kind": "Comment", "line": 2, "text": " ", "block": True, "trailing": True},
    {"kind": "ExpressionStatement", "line": 2, "expression": "++b"}]
assert parse("a; /*\n*/\nb;")[1]["text"] == "\n"  # with another line end: as tree-sitter reads it
assert value("f(a /*\n*/, b)") == "f(a, b)"  # within an expression, dropped as any comment there
assert printed("a /*\n*/ b;") == "a; /*\n*/\nb;\n"
# What tree-sitter-typescript misreads, which the converter corrects
# 21. The grouping of `as` and `satisfies`; 22. `!`; 23. `readonly`; 24. `x as A.B.C` (TS5PRS-04, TS5PRS-08 have more)
assert value("a ?? b as T", shapes=True)[2][1][0] == "tsasexpression" and value("a ?? b!", shapes=True)[2][1][0] == (
    "tsnonnullexpression")
assert first("type X = readonly A[] | B;", shapes=True)["typeAnnotation"][0] == "tsuniontype"
assert value("x as A.B.C", shapes=True)[0] == "tsasexpression"
# 25. `using` declarations with several declarators, and in a `for`'s initializer
declaration = first("await using a = f(), b = g();")
assert declaration["declarationKind"] == "await using" and [d["id"] for d in declaration["declarations"]] == ["a", "b"]
assert first("for (using a = f(), b = g(); ; ) {}")["init"]["declarations"][1]["init"] == "g()"
assert first("using a = f(), b;")["kind"] == "Error"  # each declarator needs its value
assert value("x = (using a = 1)", shapes=True)[2][1][2][0][:2] == ("error", {"text": "using a = 1"})  # no expression
assert same("using a = f(), b = g();\n") and same("for (using a = f();;) {}\n")
# 26. `static`, `get` and `set` on a line of their own
held = first("class C {\n    static\n    x = 1;\n    @d static\n    m() {}\n    get\n    g() {\n        return 1;\n    }\n"
             "    static\n    set\n    g(v) {}\n    static\n    [k: string]: number;\n    static;\n    y = 1;\n}")["body"]["body"]
assert [(m["kind"], m.get("static", False), m.get("methodKind"), m.get("line")) for m in held] == [
    ("PropertyDefinition", True, None, 2), ("MethodDefinition", True, "method", 4), ("MethodDefinition", False, "get", 6),
    ("MethodDefinition", True, "set", 10), ("TSIndexSignature", True, None, None), ("PropertyDefinition", False, None, 15),
    ("PropertyDefinition", False, None, 16)]
assert held[1]["decorators"] == [{"kind": "Decorator", "expression": "d"}] and held[5]["key"] == "static"
assert printed("class C {\n    get\n    x = 1;\n}") == "class C {\n    get;\n    x = 1;\n}\n"  # see 42
# 27. `namespace A {}`; 28. `global { ... }`; 29. `declare module "m";`; 30. `bigint` (TS5PRS-07, TS5PRS-08 have more)
assert first("x;\nnamespace A {}")["kind"] == "ExpressionStatement" and parse("x;\nnamespace A {}")[1]["kind"] == (
    "TSModuleDeclaration")
assert first("type X = bigint;", shapes=True)["typeAnnotation"][0] == "tsbigintkeyword"
assert first("global {\n}") == {"kind": "TSModuleDeclaration", "line": 1, "moduleKind": "global",
                                 "id": {"kind": "Identifier", "name": "global"}, "body": {"kind": "TSModuleBlock"}}
# 31. `<const>x`
assert value("<const>['a']", shapes=True)[2][0] == ("tstypereference", {}, [("identifier", {"name": "const"}, [])])
assert same('x = <const>["a"];\n')
# 32. `new a!.b()`; 33. `new new a()()`
assert value("new a!.b<T>(1)", shapes=True)[:2] == ("newexpression", {"typeArguments": (
    "tstypeparameterinstantiation", {})})
assert value("new a.b!()") == "new (a.b!)()" and value("new a![0]()") == "new (a![0])()"
assert value("new new a()()", shapes=True)[2][0][0] == "newexpression" and printed("new new a(1)<T>(2);") == (
    "new (new a(1))<T>(2);\n")
assert value("new (new a())()") == "new (new a())()" and value("(new a)!.b()", shapes=True)[0] == "callexpression"
# What tree-sitter-typescript reads but TypeScript rejects, kept invalid syntax
# 34. Where an enum member stands (TS5PRS-12 has it)
assert first("enum E { A = , B }")["body"]["members"] == [{"kind": "Error", "line": 1, "text": "A = , B"}]
assert first("enum E { A, = }")["body"]["members"][1] == {"kind": "Error", "line": 1, "text": "="}
# 35. An optional chain as a tag or a constructor
assert value("x = a?.b`t`", shapes=True)[2][1][:2] == ("error", {"text": "a?.b`t`"})
assert value("new a?.b()", shapes=True)[:2] == ("error", {"text": "new a?.b()"})
assert value("x = (a?.b)`t`") == "x = (a?.b)`t`" and value("new (a?.b)()") == "new (a?.b)()"
# 36. A line end after an arrow's `async`, but not before its `=>`, which TypeScript reads
assert value("x = async\n(a) => 1", shapes=True)[2][1][:2] == ("error", {"text": "async\n(a) => 1"})
assert value("x = (a)\n=> 1", shapes=True)[2][1][0] == "arrowfunctionexpression"
# 37. A `using` declaration of a pattern
assert first("using { a } = f();") == {"kind": "Error", "line": 1, "text": "using { a } = f();"}
# 38. `export default` of what has no default
for text in ("export default enum E {}", "export default const x = 1;"):
    assert first(text)["kind"] == "Error", text
assert first("export default interface I {}")["declaration"]["kind"] == "TSInterfaceDeclaration"
# 39. A hole among a call's arguments, Flow's types, an optional chain as a target (TS5PRS-12 has them)
assert first('export {a} from "m" with { type: "json"')["kind"] == "Error"  # attributes that do not parse
assert value("f<=>`x`", shapes=True)[0] == value("f<a b>`x`", shapes=True)[0] == "error"  # nor type arguments
# 40. Literals TypeScript rejects
assert value("x = 010", shapes=True)[2][1] == ("error", {"text": "010"}, [])
assert [value(f"x = {t}", shapes=True)[2][1][0] for t in ("08", "0_1", "'\\01'", "'\\8'", "`\\1`", "`\\u{110000}`")] == [
    "error"] * 6
assert value("x = '\\0'", shapes=True)[2][1][1]["value"] == "\0" and value("x = 0") == "x = 0"
assert value("x = tag`\\1`") == "x = tag`\\1`"  # a tagged template may hold any escape
assert value("x = `\\unicode`", shapes=True)[2][1][:2] == ("error", {"text": "`\\unicode`"})
# What stays as tree-sitter-typescript reads it
# 41. `await`, `yield` and `interface` as names
assert first("function f(yield) {}")["kind"] == "Error" and value("x = yield + 1", shapes=True)[2][1][0] == (
    "yieldexpression")
# 42. `get` or `set` on a line of its own before a field: two fields (26)
print("ok")